<a href="https://colab.research.google.com/github/tensorbytes0202/Transformer-v-s-Bert-from-scratch-/blob/main/BERT_vs_Transformer_(Scratch_Study).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PHASE 1:


In [46]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from collections import Counter
from sklearn.model_selection import train_test_split

In [47]:
train = pd.read_csv("/content/test.csv",
                    header=None,
                    names=["label","title","description"]
                    )
test = pd.read_csv(
    "test.csv",
    header=None,
    names=["label","title","description"]
)

print(train.shape)
print(test.shape)
train.head()

(7601, 3)
(7601, 3)


,label,title,description
0,Class Index,Title,Description
1,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
2,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
3,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
4,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...


In [48]:
train["text"] = (
    train["title"].astype(str)
    + " "
    + train["description"].astype(str)
)

test["text"] = (
    test["title"].astype(str)
    + " "
    + test["description"].astype(str)
)

train = train[["text","label"]]
test = test[["text","label"]]

train.head()

,text,label
0,Title Description,Class Index
1,Fears for T N pension after talks Unions repre...,3
2,The Race is On: Second Private Team Sets Launc...,4
3,Ky. Company Wins Grant to Study Peptides (AP) ...,4
4,Prediction Unit Helps Forecast Wildfires (AP) ...,4


In [49]:
train["length"]  =train["text"].apply(lambda x:len(str(x).split())
)

train["length"].describe()

,length
count,7601.000000
mean,37.716353
std,10.137051
min,2.000000
25%,32.000000
50%,37.000000
75%,43.000000
max,137.000000


In [50]:
train["length"].quantile(0.95)

np.float64(52.0)

In [51]:
import re

def tokenize(text):

    text = text.lower()

    text = re.sub(
        r'[^a-z0-9 ]',
        '',
        text
    )

    return text.split()

In [52]:
tokenize(
    "Hello World! AI is Amazing."
)

['hello', 'world', 'ai', 'is', 'amazing']

In [53]:
counter  = Counter()
for text in train["text"]:

  tokens = tokenize(text)
  counter.update(tokens)

word2idx = {
    "<PAD>":0,
    "<UNX":1
}

In [54]:
idx = 2

for word,count in counter.items():
  if count >=2:
    word2idx[word] = idx

    idx+=1

In [55]:
len(word2idx)

13670

In [56]:
def encode(text):

  tokens = tokenize(text)

  return [
      word2idx.get(token,1)
      for token in tokens
  ]

In [57]:
encode(
    "Artificial Intelligence"
)

[2935, 2458]

In [58]:
MAX_LEN  = 100

def pad(sequence):
  sequence = sequence[:MAX_LEN]

  sequence += [0] * (
      MAX_LEN - len(sequence)
  )

  return sequence

In [59]:
X = [
    pad(encode(text)
    )
    for text in train["text"]
]

y = train["label"].values

In [60]:
print(len(X))
print(len(X[0]))

7601
100


In [61]:
train["length"].describe()
train["length"].quantile(0.95)
len(word2idx)

13670

PHASE 2:


Hyper parameters

In [62]:
VOCAB_SIZE = len(word2idx)

EMBED_DIM = 128
NUM_HEADS = 4

FF_DIM = 512

NUM_LAYERS = 2

NUM_CLASSES = 4
MAX_LEN = 100

DROUPOUT  = 0.1

Embedding Layer

In [63]:
class TokenEmbedding(nn.Module):

  def __init__(self,vocab_size,embed_dim):

    super().__init__()

    self.embedding  = nn.Embedding(vocab_size,embed_dim,padding_idx=0)

  def forward(self,x):

    return self.embedding(x)

In [64]:
emb = TokenEmbedding(
    VOCAB_SIZE,
    EMBED_DIM
)

sample = torch.tensor(X[:2])

print(
    emb(sample).shape
)

torch.Size([2, 100, 128])


In [65]:
import math

Positional Encoding

In [66]:
class PositionalEncoding(nn.Module):

  def __init__(self,embed_dim,max_len=5000):

    super().__init__()

    pe = torch.zeros(max_len,embed_dim)

    position = torch.arange(0,max_len).unsqueeze(1)

    div_term = torch.exp(torch.arange(0,embed_dim,2)*(-math.log(10000.0)/ embed_dim)
        )

    pe[:,0::2] = torch.sin(position * div_term)

    pe[:,1::2]  =torch.cos(position * div_term)

    pe = pe.unsqueeze(0)

    self.register_buffer("pe",pe)

  def forward(self,x):
       return x + self.pe[:,:x.size(1)]

In [67]:
pos = PositionalEncoding(
    EMBED_DIM
)

out = pos(
    emb(sample)
)

print(out.shape)

torch.Size([2, 100, 128])


Multi-Head Attention


In [68]:
class MultiHeadAttention(nn.Module):

  def __init__(
      self,
      embed_dim,
      num_heads
  ):

    super().__init__()

    self.embed_dim = embed_dim
    self.num_heads = num_heads
    self.head_dim = (embed_dim // num_heads)

    self.q_proj = nn.Linear(embed_dim,embed_dim)
    self.k_proj = nn.Linear(embed_dim,embed_dim)
    self.v_proj = nn.Linear(embed_dim,embed_dim)
    self.o_proj = nn.Linear(embed_dim,embed_dim)


  def forward(self,x):

    B,L,E  = x.shape

    Q = self.q_proj(x)
    k = self.k_proj(x)
    V = self.v_proj(x)

    Q = Q.view(B,L,self.num_heads,self.head_dim
          ).transpose(1,2)

    k = k.view(B,L,self.num_heads,self.head_dim
          ).transpose(1,2)

    V = V.view(B,L,self.num_heads,self.head_dim
          ).transpose(1,2)

    scores = torch.matmul(
        Q,
        k.transpose(-2,-1)
    )

    scores = scores / (
        self.head_dim ** 0.5
    )

    attention = torch.nn.functional.softmax(
        scores,
        dim = -1)

    out = torch.matmul(
        attention,
        V
    )

    out = out.transpose(1,2).contiguous()

    out = out.view(B,L,E)

    return self.o_proj(out)

In [69]:
mha =  MultiHeadAttention(
    EMBED_DIM,
    NUM_HEADS
)

out = mha(out)

print(out.shape)


torch.Size([2, 100, 128])


Feed Forward Network

In [70]:
class FeedForward(nn.Module):

  def __init__(self,embed_dim,ff_dim):

    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(embed_dim,ff_dim),

    nn.ReLU(),

    nn.Linear(
        ff_dim,
        embed_dim
    )
    )

  def forward(self,x):
    return self.net(x)

Encoder Block


In [71]:
class EncoderBlock(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ff_dim
    ):

        super().__init__()

        self.attn = MultiHeadAttention(
            embed_dim,
            num_heads
        )

        self.norm1 = nn.LayerNorm(
            embed_dim
        )

        self.ff = FeedForward(
            embed_dim,
            ff_dim
        )

        self.norm2 = nn.LayerNorm(
            embed_dim
        )

    def forward(self,x):

        x = self.norm1(
            x + self.attn(x)
        )

        x = self.norm2(
            x + self.ff(x)
        )

        return x

In [72]:
encoder = EncoderBlock(
    EMBED_DIM,
    NUM_HEADS,
    FF_DIM
)

out = encoder(
    emb(sample)
)

print(out.shape)

torch.Size([2, 100, 128])


Phase 3

Transformer classifier


In [73]:
class TransformerClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        ff_dim,
        num_layers,
        num_classes
    ):

        super().__init__()

        self.embedding = TokenEmbedding(
            vocab_size,
            embed_dim
        )

        self.positional = PositionalEncoding(
            embed_dim,
            MAX_LEN
        )

        self.encoders = nn.ModuleList(
            [
                EncoderBlock(
                    embed_dim,
                    num_heads,
                    ff_dim
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self,x):

        x = self.embedding(x)

        x = self.positional(x)

        for encoder in self.encoders:

            x = encoder(x)

        x = x.mean(dim=1)

        x = self.dropout(x)

        return self.classifier(x)

Create Model

In [74]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = TransformerClassifier(
    VOCAB_SIZE,
    EMBED_DIM,
    NUM_HEADS,
    FF_DIM,
    NUM_LAYERS,
    NUM_CLASSES
)

model = model.to(device)

print(model)

TransformerClassifier(
  (embedding): TokenEmbedding(
    (embedding): Embedding(13670, 128, padding_idx=0)
  )
  (positional): PositionalEncoding()
  (encoders): ModuleList(
    (0-1): 2 x EncoderBlock(
      (attn): MultiHeadAttention(
        (q_proj): Linear(in_features=128, out_features=128, bias=True)
        (k_proj): Linear(in_features=128, out_features=128, bias=True)
        (v_proj): Linear(in_features=128, out_features=128, bias=True)
        (o_proj): Linear(in_features=128, out_features=128, bias=True)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ff): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=128, bias=True)
        )
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=128, out_feature

Train/Validation Split

In [75]:
# Clean the data and convert labels to 0-indexed (1-4 becomes 0-3)
mask = [str(label).isdigit() for label in y]
X_cleaned = [X[i] for i, valid in enumerate(mask) if valid]
y_cleaned = [int(y[i]) - 1 for i, valid in enumerate(mask) if valid]

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_cleaned,
    y_cleaned,
    test_size=0.2,
    random_state=42
)

Tensor Conversion

In [76]:
X_train = torch.as_tensor(X_train, dtype=torch.long)
X_val = torch.as_tensor(X_val, dtype=torch.long)
y_train = torch.as_tensor(y_train, dtype=torch.long)
y_val = torch.as_tensor(y_val, dtype=torch.long)

DataLoader

In [77]:
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

train_dataset = TensorDataset(
    X_train,
    y_train
)

val_dataset = TensorDataset(
    X_val,
    y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64
)

In [78]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

Training Function

In [79]:
def train_epoch():

    model.train()

    total_loss = 0

    correct = 0

    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (
            preds == y_batch
        ).sum().item()

        total += y_batch.size(0)

    return (
        total_loss / len(train_loader),
        correct / total
    )

In [80]:
def evaluate():

    model.eval()

    correct = 0

    total = 0

    with torch.no_grad():

        for X_batch,y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            preds = outputs.argmax(1)

            correct += (
                preds == y_batch
            ).sum().item()

            total += y_batch.size(0)

    return correct / total

In [81]:
EPOCHS = 10

In [82]:
for epoch in range(EPOCHS):

    train_loss, train_acc = train_epoch()

    val_acc = evaluate()

    print(
        f"Epoch {epoch+1}"
        f" | Loss={train_loss:.4f}"
        f" | Train Acc={train_acc:.4f}"
        f" | Val Acc={val_acc:.4f}"
    )

Epoch 1 | Loss=1.3933 | Train Acc=0.2701 | Val Acc=0.2513
Epoch 2 | Loss=1.3476 | Train Acc=0.3488 | Val Acc=0.3928
Epoch 3 | Loss=1.1706 | Train Acc=0.5066 | Val Acc=0.5618
Epoch 4 | Loss=0.9091 | Train Acc=0.6428 | Val Acc=0.6382
Epoch 5 | Loss=0.7442 | Train Acc=0.7173 | Val Acc=0.6836
Epoch 6 | Loss=0.6325 | Train Acc=0.7640 | Val Acc=0.7145
Epoch 7 | Loss=0.5539 | Train Acc=0.7942 | Val Acc=0.7401
Epoch 8 | Loss=0.4814 | Train Acc=0.8260 | Val Acc=0.7447
Epoch 9 | Loss=0.4324 | Train Acc=0.8474 | Val Acc=0.7533
Epoch 10 | Loss=0.3820 | Train Acc=0.8658 | Val Acc=0.7421
